# Analyse post preprocessing

## Objectif
L'objectif de cette pré-analyse est de se rendre compte des données en notre possession et de leur validité. 

### Étapes
Pour commencer on va vérifier que notre colonne des témoignages contient des valeurs textuelles et non des float, entier, ... 

In [1]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

In [1]:
import pandas as pd

try:
    df = pd.read_csv("../../Student_Dataset.csv") # Ouvre le csv
except FileNotFoundError:
    print("Fichier non trouve")
    raise SystemExit(84)

dfStr = df.iloc[:, 1]

print(dfStr.dtypes)

str


In [2]:
print(df.iloc[:, 0].head(15))

2                                              0x000000
3                                              0x000000
4     A short sprint brought on a twinge in the midd...
5                                              0x000000
6     A sudden twist while reaching for a bag made o...
7                                              0x000000
8     I noticed a rash on my forearms after handling...
9                                              0x000000
10                                             0x000000
11    I feel an unexplained pressure behind my eyes ...
12                                             0x000000
13    There’s a small sore area on the top of my foo...
14    I have an increased thirst level and I’m going...
15                                             0x000000
16                                             0x000000
Name: 1, dtype: str


On voit donc que notre colonne numéro deux qui contient les témoignages n'a que des string et aucun autre type

On va donc vérifier les NaN potentiels

In [3]:
print(dfStr[dfStr.isna()])

4       NaN
6       NaN
8       NaN
11      NaN
13      NaN
       ... 
1000    NaN
1001    NaN
1003    NaN
1005    NaN
1010    NaN
Name: On uneven paths the heel feels wobbly side‑to‑side. Slanted pavements bring a quick aching shift under the ankle when I step., Length: 396, dtype: str


On comprend un problème, comme nous n'avons pas de label la colonne 2 se mélange avec `0x000000` et le témoignage.
Pour aller plus loin dans l'analyse on va donc retirer tous les `0x000000`, ils n'apportent rien aux témoignages, on va donc les retirer pour avoir une deuxième colonne parfaitement conforme et savoir si les témoignages contiennent des NaN.

In [4]:
dfStr = dfStr.str.replace('0x000000,', '', regex=False)
print(dfStr)

2       The feverish feeling eased, but I’m now breath...
3       While working in the garden, I touched a metal...
4                                                     NaN
5       I accidentally left my hand on a hot mug for a...
6                                                     NaN
                              ...                        
1007    I slammed my fingertip in a drawer. The tip is...
1008    I woke up with a throbbing pain in my left ear...
1009    During the summer, my throat gets itchy and I ...
1010                                                  NaN
1011    Every morning I wake up with a pounding headac...
Name: On uneven paths the heel feels wobbly side‑to‑side. Slanted pavements bring a quick aching shift under the ankle when I step., Length: 1010, dtype: str


In [5]:
try:
    df = pd.read_csv('../../Student_Dataset.csv')
except FileNotFoundError:
    print("Fichier non trouve")
    raise SystemExit(84)

# Si la colonne 1 est vide, le texte est dans la colonne 0. Sinon on prend le 1
textes_propres = np.where(df.iloc[:, 1].isna(), df.iloc[:, 0], df.iloc[:, 1])

# on génère un tableau clean avec deux colonnes
df_final = pd.DataFrame({
    'Chiffre': range(1, len(df) + 1),
    'Temoignage': textes_propres
})


print(df_final.head(10))

   Chiffre                                         Temoignage
0        1  The feverish feeling eased, but I’m now breath...
1        2  While working in the garden, I touched a metal...
2        3  A short sprint brought on a twinge in the midd...
3        4  I accidentally left my hand on a hot mug for a...
4        5  A sudden twist while reaching for a bag made o...
5        6  During a birthday party, a hot beverage was kn...
6        7  I noticed a rash on my forearms after handling...
7        8  Since I lost 20 pounds, my headaches have less...
8        9  The rash on my face and jaw is a classic hives...
9       10  I feel an unexplained pressure behind my eyes ...


On vient donc de régler nos problèmes de colonnes et de data, on a donc créé un nouveau tableau propre qui ne contient plus les `0x000000`. De plus nous avons ajouté des labels pour plus de simplicité.
On peut donc maintenant vérifier si un NaN est présent.

In [6]:
print(df_final.isna().sum())

Chiffre       0
Temoignage    0
dtype: int64


On obtient donc 0 dans chiffre et témoignage donc nous n'avons aucune valeur NaN

Nous allons faire une dernière vérification, si le témoignage est inférieur à 10 caractères on va considérer que le témoignage n'est pas complet/ pertinent car trop petit

In [7]:
df_final = df_final[df_final['Temoignage'].str.len() > 10]

Nous avons donc nettoyé les témoignages de moins de 10 caractères.

##### Vérification du nombre de doublons

In [8]:
print("Nombre de témoignages en double :", df_final.duplicated(subset=["Temoignage"]).sum())

Nombre de témoignages en double : 9


Nous observons 15 doublons. Nous n'avons pas la possibilité de savoir si ces témoignages sont des duplications ou simplement des témoignages identiques.
Pour garder l'intégrité du dataset nous allons les supprimer, car ils ne représentent qu'une infime partie.

Après réalisation de graphique, nous avons des apostrophes qui ne sont pas sous forme classique, nous allons donc les changer en `'` pour normaliser le texte.

In [9]:
df_final['Temoignage'] = df_final['Temoignage'].str.replace('’', "'")
raise SystemExit(0)

Les apostrophes ont bien été changées avec succès.
L'analyse du CSV est donc terminée, nous avons toutes les informations nécessaires pour réaliser notre nouveau CSV propre ()

#### On va regarder les images similaires 
Notre dataset est un regroupement de plusieurs scanners d'hôpitaux au vu des résultats sur les premiers modèles il va falloir vérifier dans le `.npy` si nous n'avons pas des images dupliquées qui ont une matrice similaire.



In [15]:
# on charge notre fichier de matrice
try:
    matrices = np.load('../../Matrice.npy', allow_pickle=True)
except FileNotFoundError:
    print("Fichier non trouve")
    raise SystemExit(84)

# on aplatit les matrices pour les comparer + facilement
flat = matrices.reshape(matrices.shape[0], -1)

# mesure la direction entre deux valeurs (images) et donne un chiffre de similitude
sim = cosine_similarity(flat)
# on trouve les similaires mais on prend l'image elle-même
np.fill_diagonal(sim, 0)

prctRessemblance = 0.995
paris_P_Identique = np.argwhere(sim > prctRessemblance)

print(f"Nombre de paires quasi identique : {len(paris_P_Identique)//2}")

Nombre de paires quasi identique : 6142


On charge notre matrice et on fait un cosine similarity pour savoir combien d'images ont un doublon.

On doit ensuite établir les groupes d'images similaires.

In [14]:
import numpy as np

def find(parent, i):
        """
        Permet de remonter dans tous les parents jusqu'à se trouver elle-même.
        Si on a [0, 0, 1, 4]
        on cherche find(parent, 1), on obtient 0 donc on fait find(parent, 0) et on obtient 0 (la même image que le i) donc on a terminé on est retombé dessus, alors 1 et 0 sont les mêmes

        Prend en argument parent et i:
            - i est l'image qu'on cherche
            - parent est une liste qui contient globalement qui pointe vers qui, imaginons image A pointe vers l'image B alors dans parent on aura B, et ça se cumule avec les autres enfants

        return i
        """
        # tant qu'on est pas sur le même
        while parent[i] != i:
            i = parent[i]
        return i

def union(parent, i, j):
    """
    Prend en paramètre parent i et j:
        - parent est la liste qui contient qui pointe vers qui
        - i et j sont les images qu'on veut fusionner
    """
    lst_i = find(parent, i)
    list_j = find(parent, j)
    # on vérifie qu'ils sont pas déjà dans le même groupe
    if lst_i != list_j:
        # on fait pointer i vers j qui devient son parent
        parent[list_j] = lst_i


# on prend la première colonne
n = sim.shape[0]

# crée le tableau parent de base, au début chaque image pointe vers elle-même car elle est identique à elle-même
parent = np.arange(n)


# on va faire des paires d'images similaires
pairs = np.argwhere(sim > prctRessemblance)
# (voir en bas pour l'explication)

# on passe dans chacune des paires pour trouver les unions avec les autres paires
for i, j in pairs:
    union(parent, i, j)


# on va stocker tt les résultats 
resultat = []
for i in range(n):
    resultat.append(find(parent, i))


nbr_groupes = len(np.unique(resultat))
print(f"Nombre de groupes donc patients : {nbr_groupes}")
print(f"Taille moyenne d'1 groupe : {n / nbr_groupes:.1f} images")

Nombre de groupes donc patients : 6885
Taille moyenne d'1 groupe : 1.8 images


##### Explication de argwhere et du système actuel pour les images similaires

Notre système se base sur les matrices, imaginons que nous avons au total 4 images :

```
    [0.0,  0.99, 0.2,  0.998],
    [0.99, 0.0,  0.1,  0.3],
    [0.2,  0.1,  0.0,  0.15],
    [0.998,0.3,  0.15, 0.0]
```

on a donc une matrice de 4x4 avec des 0 en diagonale car on a dit `np.fill_diagonal(sim, 0)` donc colonne 1 ligne 1 qui est l'image 1 ne peut pas être identique à 1.

Ensuite on fait sim > prctRessemblance donc on passe dans chacun et on regarde si la valeur est > à 0.995.
Prenons colonne 1 ligne 0 on a donc 0.99 donc c'est < à 0.995 alors on va mettre false à la place.

On fait ça pour tout et on obtient :

```
    [False, False, False, True],
    [False, False, False, False],
    [False, False, False, False],
    [True,  False, False, False]
```

Donc on a ligne 0 colonne 4 et colonne 0 ligne 4, nous avons donc trouvé une paire [0, 4] donc ces deux images seront similaires 


On passe ensuite dans les paires et on essaye de trouver les similaires et par ricochet ça nous donnera toutes les similaires.


##### Pourquoi faire ça ?

Comme notre dataset contient du dataleak car le dataset contient plusieurs scanners d'hôpitaux, certaines personnes ont fait plusieurs scanners dans le même ou un différent hôpital.
Il y a donc des images du même patient approximativement 1.8 par patient (moyenne d'un groupe).

Donc pour éviter de fausser nos données on ne va pas inclure pour un même patient une de ces images dans l'entraînement et 1 dans le test car sinon il va trouver directement sans chercher, il répondra car il l'a appris par cœur.
Donc on envoie toutes les images d'un patient dans 1 seule des deux parties seulement (entraînement ou test) et c'est tout.